# Track 1 · 13 CNN on the raw EEG image — a simple model on the same Alljoined data

**One job.** Treat each EEG window (electrodes x time, 120 Hz) as an image and fine-tune a CNN on it,
against the same DINOv2-giant targets, the same 5 held-out people and the same score as
`track1_11` (REVE: held-out people 5.04% per window / 8.90% subject-aggregated top-5, chance 2.5%).
It is a cheap probe: what can a simple image model do on the sealed-style data? Needs
`track1_12_alljoined_prepare` to have run (same windows). Self-contained.

## Models (`MODEL`)

- `'resnet101_lora'` (default; also `'resnet18_lora'`, `'resnet50_lora'`): the torchvision ResNet with ImageNet
  weights, the first convolution summed to one input channel. Frozen convolutions with a **low-rank (LoRA) update beside every
  convolution**; BatchNorm and the head train. Before export the update is merged into plain
  convolution weights, so the submission is an ordinary ResNet.
- `'small_cnn'`: four conv blocks from scratch (control; in Track 2 a small CNN from scratch beat a
  pretrained ResNet-50 on PSD maps, 0.519 vs 0.416 mix).

Head: Dropout, Linear to 64, GELU, Dropout, Linear to 1536 (the same funnel width as track1_11).
Input: the raw window, standardised per window; no PSD (a PSD over a short window loses timing, and
the sealed harness only ever sends 1 s windows). Rows are electrodes in the trained order.

## Aim: generalisation to new people (beating chance), not a high score on this data

- The epoch is chosen, and training stopped, on **whole people who are never fitted** (`N_VAL_PEOPLE`, on
  images held out for everyone), not on seen people's new images.
- First run (rank 2, dropout 0.6 twice, 64-wide head, low learning rates, weight decay 0.05) never left chance:
  training loss 5.5568 to 5.5454 against chance 5.5452, validation rank 0.50. These defaults are the loosened
  version: rank 16, learning rates 1e-3 / 2e-3, dropout 0.2, no weight decay, a 512-wide head, gentle
  augmentation, adapters training from epoch 1, lenient early stopping (patience 5), and a **40,000-window
  fit subset** (`MAX_FIT_WINDOWS`, `None` = all) so an epoch takes minutes. After epoch 2 the notebook prints a
  *learning check*: whether the training loss is below chance. The `gap` column is validation loss minus
  training loss (training loss is on augmented views, so it runs high).
- Augmentation: a time-shifted noisy copy and a copy with the electrode rows shuffled and a few electrodes
  dropped (the time axis inside a row is never shuffled).

## Training (as track1_11)

Three views per window, mean loss (see above). InfoNCE (temperature 0.07). Best validation-rank epoch kept and saved to
Drive as it improves; early stop on a plateau or a rising validation loss. Epoch-1 time and a
projected total are printed within the first 100 batches.


## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'
DATA = 'alljoined'               # Track1/<DATA>/ written by track1_12_alljoined_prepare

MODEL = 'resnet101_lora'         # 'resnet18_lora', 'resnet50_lora', 'resnet101_lora' (ImageNet weights + LoRA),
                                 # or 'small_cnn' (from scratch, control)
ARCH = 'small' if MODEL == 'small_cnn' else MODEL.replace('_lora', '')

N_HOLDOUT_PEOPLE = 5             # whole people never fitted or validated; scored at the end (same 5 as track1_11)
N_VAL_PEOPLE = 3                 # whole people kept out of fitting to choose the epoch and stop early on
                                 # (unseen people, unseen images). 0: validate on seen people's new images
HOLDOUT_PEOPLE = []              # or name them (subject ids as in the file names, e.g. ['07', '08'])

EPOCHS = 30                      # upper bound; early stopping decides
HEAD_ONLY_EPOCHS = 0             # LoRA adapters stay frozen for this many epochs (0: adapters train from epoch 1)
MAX_FIT_WINDOWS = 40000          # fit on a random subset of this many windows so an epoch takes minutes
                                 # (a learnability check). None = all training windows
BATCH = 256
LR_ADAPTER, LR_HEAD = 1e-3, 2e-3   # adapters; head + BatchNorm (all weights for small_cnn)
LORA_RANK, LORA_ALPHA = 16, 16   # on every convolution of the ResNet
SHRINK = 512                     # head funnel width (wide: 64 starved the first run)
DROPOUT = 0.2
WEIGHT_DECAY = 0.0               # AdamW, on the head, BatchNorm and adapters
TEMPERATURE = 0.07
USE_AMP = True                   # mixed precision on GPU

AUGMENT = True                   # 3 views per window (False: originals only, control):
                                 # original | time-shifted + noisy | row-shuffled + electrodes dropped
NOISE_FRAC = 0.1
SHIFT_SAMPLES = 3                # random time shift of up to +-3 samples (25 ms) in the noisy view
MAX_DROP_ELECTRODES = 2          # up to this many random electrodes zeroed in the shuffled view
KEEP = 'best'                    # 'best' validation-rank epoch (also saved to Drive as it happens), or 'last'
PATIENCE = 5
MIN_DELTA = 0.0005
DIVERGE_LOSS = 0.15

MAKE_ZIP = True
VAL_IMAGE_FRAC = 0.2
SEED = 33

RUN_TAG = (f'cnn_{MODEL}_s{SHRINK}_d{DROPOUT}_b{BATCH}'
           + (f'_r{LORA_RANK}a{LORA_ALPHA}' if MODEL != 'small_cnn' else '')
           + (f'_val{N_VAL_PEOPLE}p' if N_VAL_PEOPLE else '')
           + (f'_fit{MAX_FIT_WINDOWS}' if MAX_FIT_WINDOWS else '')
           + (f'_aug{NOISE_FRAC:g}' if AUGMENT else '_noaug') + f'_{KEEP}')
RUN_TRAINING = True
print(f'NOTEBOOK VERSION: track1_13 v1   settings {RUN_TAG}   epochs <= {EPOCHS}   '
      f'chance loss will be {__import__("math").log(BATCH):.4f}')


In [ ]:
import importlib
for mod in ('torch', 'torchvision'):
    print(f'{mod:12} {getattr(importlib.import_module(mod), "__version__", "ok")}')


In [ ]:
import json, os, shutil, sys, time, types, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, torch, torch.nn.functional as F

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
BASE = ROOT / 'Track1'
AJ = BASE / DATA
WEIGHTS = BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', '/content/hf')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = USE_AMP and DEVICE == 'cuda'
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)


def l2(a):
    return a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-12)


def _truth(y_ids):
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    return gal, np.array([pos[y] for y in y_ids])


def rank_pct(pred, y_ids, emb):
    gal, truth = _truth(y_ids)
    sims = l2(pred) @ l2(emb[gal]).T
    return float((sims < sims[np.arange(len(sims)), truth][:, None]).mean())


def top5(pred, y_ids, emb):
    gal, truth = _truth(y_ids)
    sims = l2(pred) @ l2(emb[gal]).T
    k = min(5, len(gal))
    tk = np.argpartition(-sims, kth=k - 1, axis=1)[:, :k]
    return 100.0 * np.mean([t in r for t, r in zip(truth, tk)]), 100.0 * k / len(gal)


def rank_null(pred, y_ids, n=200):
    """Validation rank under permuted labels, for these exact predictions."""
    gal, truth = _truth(y_ids)
    sims = l2(pred) @ l2(emb[gal]).T
    rows = np.arange(len(sims))
    return np.array([float((sims < sims[rows, p][:, None]).mean())
                     for p in (rng.permutation(truth) for _ in range(n))])


print(f'device {DEVICE}, mixed precision {AMP}')

## Step 1 — the Alljoined windows and targets

In [ ]:
t = np.load(AJ / 'image_embeddings.npz', allow_pickle=True)
emb = l2(t['emb'].astype('float32'))
DIM = emb.shape[1]
key = {k: i for i, k in enumerate(t['keys'].tolist())}

train_files = sorted((AJ / 'windows').glob('*_train.npz'))
test_files = sorted((AJ / 'windows').glob('*_test.npz'))
if not train_files or not test_files:
    raise FileNotFoundError(f'no windows in {AJ / "windows"}; run track1_12_alljoined_prepare first')

# Pass 1: labels, people and which windows each file contributes (small).
plan, ys, subs, CH = [], [], [], None
for f in train_files:
    z = np.load(f, allow_pickle=True)
    idx = np.array([key.get(k_, -1) for k_ in z['keys'].tolist()])
    keep = idx >= 0
    if CH is None:
        CH = list(z['channels'])
    elif list(z['channels']) != CH:
        raise RuntimeError(f'{f.name}: channel order differs from the first training file')
    plan.append((f, keep))
    ys.append(idx[keep])
    subs.append(np.full(int(keep.sum()), str(z['subject'])))
n_train = sum(int(k.sum()) for _, k in plan)

# Pass 2: the windows themselves, written once into one array on the machine's local disk
# (not Drive) and read per batch; a rerun in the same session reuses it.
local = Path('/content') if Path('/content').exists() else AJ
xfile, sigfile = local / 'track1_aj_train_X.npy', local / 'track1_aj_train_X.json'
signature = json.dumps([[str(f), f.stat().st_size, int(k.sum())] for f, k in plan])
if xfile.exists() and sigfile.exists() and sigfile.read_text() == signature:
    Xtr = np.load(xfile, mmap_mode='r')
    print(f'reusing {xfile} ({xfile.stat().st_size / 1e9:.1f} GB)')
else:
    Xtr, i, t0 = None, 0, time.time()
    for f, keep in plan:
        x = np.load(f)['X'][keep]
        if Xtr is None:
            Xtr = np.lib.format.open_memmap(xfile, mode='w+', dtype='float16',
                                            shape=(n_train,) + x.shape[1:])
        Xtr[i:i + len(x)] = x
        i += len(x)
        del x
    Xtr.flush()
    sigfile.write_text(signature)
    Xtr = np.load(xfile, mmap_mode='r')
    print(f'{n_train:,} training windows from {len(plan)} files written to {xfile} '
          f'in {time.time() - t0:.0f}s')
Wtr = [Xtr, np.concatenate(ys), np.concatenate(subs)]

parts = [[], [], []]
for f in test_files:
    z = np.load(f, allow_pickle=True)
    idx = np.array([key.get(k_, -1) for k_ in z['keys'].tolist()])
    keep = idx >= 0
    parts[0].append(z['X'][keep]); parts[1].append(idx[keep])
    parts[2].append(np.full(int(keep.sum()), str(z['subject'])))
    if list(z['channels']) != CH:
        raise RuntimeError(f'{f.name}: test channel order differs from training')
Wte = [np.concatenate(a) for a in parts]
IN_SFREQ = float(json.loads((AJ / 'manifest.json').read_text())['sfreq'])

people = sorted(set(Wtr[2].tolist()))
if not HOLDOUT_PEOPLE:
    HOLDOUT_PEOPLE = sorted(rng.choice(people, N_HOLDOUT_PEOPLE, replace=False).tolist())
HOLDOUT_PEOPLE = [str(p_) for p_ in HOLDOUT_PEOPLE]
assert set(HOLDOUT_PEOPLE) <= set(people), f'unknown people {set(HOLDOUT_PEOPLE) - set(people)}'
SEEN = ~np.isin(Wtr[2], HOLDOUT_PEOPLE)        # rows allowed into fit AND val
seen_imgs = np.unique(Wtr[1][SEEN])
val_imgs = set(rng.choice(seen_imgs, int(len(seen_imgs) * VAL_IMAGE_FRAC), replace=False).tolist())
IV = np.array([y in val_imgs for y in Wtr[1]])
# Choose the epoch on whole people who are not fitted (unseen people, unseen images): the point of
# this run is generalisation to new people, and seen people's new images would not measure that.
others = [p_ for p_ in people if p_ not in HOLDOUT_PEOPLE]
VAL_PEOPLE = sorted(rng.choice(others, N_VAL_PEOPLE, replace=False).tolist()) if N_VAL_PEOPLE else []
VALP = np.isin(Wtr[2], VAL_PEOPLE)
FIT = ~IV & SEEN & ~VALP
VAL = (IV & VALP) if N_VAL_PEOPLE else (IV & SEEN)
print(f'{len(people)} people; HELD OUT {HOLDOUT_PEOPLE}: {(~SEEN).sum():,} windows never fitted or validated')
print(f'train {Wtr[0].shape}   test {Wte[0].shape}   {len(CH)} channels at {IN_SFREQ:g} Hz')
print(f'VALIDATION people {VAL_PEOPLE or "(none: seen people, new images)"}')
print(f'{FIT.sum():,} fit / {VAL.sum():,} val windows, {len(np.unique(Wte[1]))} test images')


## Step 2 — the model, from the submission file

The submission file is executed here to get the network classes, so what is trained is exactly what is submitted.

In [ ]:
# Source of truth: solvers/track1_image_reve_lora_perchan.py in the repo.
SUBMISSION_PY = '"""Track 01 submission: a CNN that reads the raw EEG window as an image.\n\nThe window (electrodes x time, 120 Hz) is standardised per window and fed to either a ResNet18\n(ResNet18/50/101: ImageNet weights adapted to one input channel, then fine-tuned with LoRA on the\nconvolutions, merged into plain weights before export) or a small CNN trained from scratch. A small head maps\nthe pooled features to the 1536-d target embedding. Nothing depends on an install at evaluation\nbeyond torch: the networks are plain torch modules defined here, with ResNet18 parameter names\nidentical to torchvision\'s, so the pretrained weights load by name.\n\nInputs are matched to the electrodes the model was trained on by name (case-insensitive); extra\nelectrodes are ignored and missing ones are zero-filled. Row order in the image is the trained\norder, so the same window gives the same output whatever order the harness sends it in.\n\nThis file is also the model definition the training notebook uses, so the two cannot drift apart.\n"""\n\nimport json\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom benchmark_utils.base_solver import CompetSolver\n\nTARGET_SFREQ = 120.0\nCLIP_SD = 15.0\n\n\ndef resample_fft(x, n_out):\n    """Resample along time via the rfft spectrum (pure torch; nothing is installed at evaluation)."""\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(*spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device)\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\nclass BasicBlock(nn.Module):\n    expansion = 1\n\n    def __init__(self, cin, cout, stride):\n        super().__init__()\n        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)\n        self.bn1 = nn.BatchNorm2d(cout)\n        self.relu = nn.ReLU(inplace=True)\n        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)\n        self.bn2 = nn.BatchNorm2d(cout)\n        self.downsample = None\n        if stride != 1 or cin != cout:\n            self.downsample = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False),\n                                            nn.BatchNorm2d(cout))\n\n    def forward(self, x):\n        idt = x if self.downsample is None else self.downsample(x)\n        out = self.relu(self.bn1(self.conv1(x)))\n        out = self.bn2(self.conv2(out))\n        return self.relu(out + idt)\n\n\nclass Bottleneck(nn.Module):\n    expansion = 4\n\n    def __init__(self, cin, planes, stride):\n        super().__init__()\n        cout = planes * 4\n        self.conv1 = nn.Conv2d(cin, planes, 1, bias=False)\n        self.bn1 = nn.BatchNorm2d(planes)\n        self.conv2 = nn.Conv2d(planes, planes, 3, stride, 1, bias=False)\n        self.bn2 = nn.BatchNorm2d(planes)\n        self.conv3 = nn.Conv2d(planes, cout, 1, bias=False)\n        self.bn3 = nn.BatchNorm2d(cout)\n        self.relu = nn.ReLU(inplace=True)\n        self.downsample = None\n        if stride != 1 or cin != cout:\n            self.downsample = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False),\n                                            nn.BatchNorm2d(cout))\n\n    def forward(self, x):\n        idt = x if self.downsample is None else self.downsample(x)\n        out = self.relu(self.bn1(self.conv1(x)))\n        out = self.relu(self.bn2(self.conv2(out)))\n        out = self.bn3(self.conv3(out))\n        return self.relu(out + idt)\n\n\nRESNETS = {"resnet18": (BasicBlock, (2, 2, 2, 2), 512),\n           "resnet50": (Bottleneck, (3, 4, 6, 3), 2048),\n           "resnet101": (Bottleneck, (3, 4, 23, 3), 2048)}\n\n\nclass ResNet(nn.Module):\n    """torchvision\'s resnet18/50/101 without the classifier, one input channel; same parameter names."""\n\n    def __init__(self, arch="resnet18"):\n        super().__init__()\n        block, layers, self.feat_dim = RESNETS[arch]\n        self.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)\n        self.bn1 = nn.BatchNorm2d(64)\n        self.relu = nn.ReLU(inplace=True)\n        self.maxpool = nn.MaxPool2d(3, 2, 1)\n        cin = 64\n        for i, (planes, n) in enumerate(zip((64, 128, 256, 512), layers), 1):\n            stride = 1 if i == 1 else 2\n            blocks = []\n            for j in range(n):\n                blocks.append(block(cin, planes, stride if j == 0 else 1))\n                cin = planes * block.expansion\n            setattr(self, f"layer{i}", nn.Sequential(*blocks))\n\n    def forward(self, x):\n        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))\n        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))\n        return x.mean(dim=(2, 3))\n\n\nclass SmallCNN(nn.Module):\n    """Four conv blocks (32, 64, 128, 256), from scratch."""\n    feat_dim = 256\n\n    def __init__(self):\n        super().__init__()\n        layers, cin = [], 1\n        for cout in (32, 64, 128, 256):\n            layers += [nn.Conv2d(cin, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout),\n                       nn.GELU(), nn.MaxPool2d(2, ceil_mode=True)]\n            cin = cout\n        self.features = nn.Sequential(*layers)\n\n    def forward(self, x):\n        return self.features(x).mean(dim=(2, 3))\n\n\nclass EEGImageNet(nn.Module):\n    def __init__(self, channels, n_outputs, in_sfreq, arch="resnet50", shrink=64, dropout=0.4):\n        super().__init__()\n        self.in_sfreq = float(in_sfreq)\n        self.backbone = ResNet(arch) if arch in RESNETS else SmallCNN()\n        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(self.backbone.feat_dim, shrink),\n                                  nn.GELU(), nn.Dropout(dropout), nn.Linear(shrink, n_outputs))\n\n    def forward(self, X):                          # (B, n_channels in trained order, T)\n        X = X.float()\n        n_out = max(int(round(X.shape[-1] * TARGET_SFREQ / self.in_sfreq)), 8)\n        X = resample_fft(X, n_out)\n        X = (X - X.mean(dim=(1, 2), keepdim=True)) / (X.std(dim=(1, 2), keepdim=True) + 1e-8)\n        X = torch.clamp(X, -CLIP_SD, CLIP_SD)\n        return self.head(self.backbone(X[:, None]))\n\n\nclass NameMatched:\n    """Reorders incoming electrodes into the trained order, zero-filling gaps."""\n\n    def __init__(self, model, channels, in_names, device):\n        self.model, self.device = model, device\n        lookup = {n.casefold(): i for i, n in enumerate(in_names)}\n        self.src = [lookup.get(c.casefold(), -1) for c in channels]\n        missing = [c for c, i in zip(channels, self.src) if i < 0]\n        print(f"[CNN] matched {len(channels) - len(missing)}/{len(channels)} training channels by "\n              f"name among {len(in_names)} inputs; zero-filled: {missing if missing else \'none\'}")\n\n    @torch.no_grad()\n    def predict(self, X):\n        X = torch.as_tensor(X, dtype=torch.float32).to(self.device)\n        out = torch.zeros(X.shape[0], len(self.src), X.shape[-1], device=self.device)\n        for dst, src in enumerate(self.src):\n            if src >= 0:\n                out[:, dst] = X[:, src]\n        self.model.eval()\n        return self.model(out)\n\n\nclass Solver(CompetSolver):\n\n    name = "EEG-image-CNN"\n\n    def load_model(self, meta):\n        sub = meta["submission_dir"]\n        cfg = json.loads((sub / "cnn_config.json").read_text())\n        in_names = meta.get("ch_names") or [c["ch_name"] for c in meta["chs_info"]]\n        model = EEGImageNet(cfg["channels"], cfg["n_outputs"], meta["sfreq"], arch=cfg["arch"],\n                            shrink=cfg["shrink"], dropout=cfg["dropout"])\n        state = torch.load(sub / "weights.pt", map_location=meta["device"], weights_only=True)\n        missing, unexpected = model.load_state_dict(state, strict=False)\n        if unexpected or missing:\n            raise RuntimeError(f"weights mismatch: missing {missing[:4]}, unexpected {unexpected[:4]}")\n        return NameMatched(model.to(meta["device"]).eval(), cfg["channels"], in_names,\n                           meta["device"])\n'

stub = types.ModuleType('benchmark_utils.base_solver')
stub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
sys.modules.setdefault('benchmark_utils', types.ModuleType('benchmark_utils'))
sys.modules['benchmark_utils.base_solver'] = stub
ns = {}
exec(compile(SUBMISSION_PY, 'submission.py', 'exec'), ns)


In [ ]:
import torch.nn as nn

class LoRAConv2d(nn.Module):
    """Frozen conv plus a rank-r update: a conv with the same kernel/stride to r channels, then a 1x1
    conv back. B starts at zero, so training begins at exactly the pretrained function. The update
    merges exactly into the kernel: W + scale * sum_r B[o, r] A[r, i, :, :]."""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.A = nn.Conv2d(base.in_channels, r, base.kernel_size, stride=base.stride,
                           padding=base.padding, bias=False)
        self.B = nn.Conv2d(r, base.out_channels, 1, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=5 ** 0.5)
        nn.init.zeros_(self.B.weight)
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + self.scale * self.B(self.A(x))

    def merged_weight(self):
        delta = torch.einsum('or,rikl->oikl', self.B.weight[:, :, 0, 0], self.A.weight)
        return self.base.weight + self.scale * delta


def add_lora(module, r, alpha):
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Conv2d):
            setattr(module, name, LoRAConv2d(child, r, alpha))
            n += 1
        else:
            n += add_lora(child, r, alpha)
    return n


def to_plain(model):
    """Copy of the model with every LoRAConv2d replaced by one conv holding the merged kernel."""
    import copy
    plain = copy.deepcopy(model)
    def walk(m):
        for name, child in list(m.named_children()):
            if isinstance(child, LoRAConv2d):
                conv = child.base
                conv.weight = nn.Parameter(child.merged_weight().detach().clone(), requires_grad=False)
                setattr(m, name, conv)
            else:
                walk(child)
    walk(plain)
    return plain


if RUN_TRAINING:
    model = ns['EEGImageNet'](CH, DIM, IN_SFREQ, arch=ARCH, shrink=SHRINK, dropout=DROPOUT)
    if MODEL != 'small_cnn':
        import torchvision
        tv = getattr(torchvision.models, ARCH)(weights='DEFAULT')   # ImageNet weights
        sd = {k: v for k, v in tv.state_dict().items() if not k.startswith('fc.')}
        sd['conv1.weight'] = sd['conv1.weight'].sum(1, keepdim=True)   # RGB -> one channel
        res = model.backbone.load_state_dict(sd, strict=True)
        print(f'ImageNet {ARCH} loaded into the backbone: {res}')
        n_lora = add_lora(model.backbone, LORA_RANK, LORA_ALPHA)
        adapters = [p for n, p in model.named_parameters() if n.endswith(('.A.weight', '.B.weight'))]
        bn_params = [p for m in model.backbone.modules() if isinstance(m, nn.BatchNorm2d)
                     for p in m.parameters()]
        head_params = list(model.head.parameters()) + bn_params
        print(f'LoRA on {n_lora} convolutions')
    else:
        adapters, head_params = [], list(model.parameters())
    model = model.to(DEVICE)
    n_ad, n_hd = sum(p.numel() for p in adapters), sum(p.numel() for p in head_params)
    n_all = sum(p.numel() for p in model.parameters())
    print(f'trainable: adapters {n_ad / 1e6:.2f}M + head/BatchNorm {n_hd / 1e6:.2f}M of {n_all / 1e6:.1f}M total')
    sample = torch.randn(2, len(CH), 120, device=DEVICE)
    print('forward check', tuple(model.eval()(sample).shape), f'(expected (2, {DIM}))')


## Step 3 — train

Three views per window, mean loss. Watch the gap between train and validation loss, and epoch 1's time (projected total printed).

In [ ]:
if RUN_TRAINING:
    EMB_T = torch.tensor(emb, device=DEVICE)


    def clip_loss(pred, idx):
        logits = (F.normalize(pred.float(), dim=-1)
                  @ F.normalize(EMB_T[idx], dim=-1).T) / TEMPERATURE
        same = idx[:, None] == idx[None, :]
        same.fill_diagonal_(False)
        logits = logits.masked_fill(same, -1e4)
        return F.cross_entropy(logits, torch.arange(len(pred), device=pred.device))


    def time_shift(x):
        B, C, T = x.shape
        sh = torch.randint(-SHIFT_SAMPLES, SHIFT_SAMPLES + 1, (B, 1, 1), device=x.device)
        idx = (torch.arange(T, device=x.device)[None, None, :] - sh).clamp(0, T - 1).expand(B, C, T)
        return x.gather(2, idx)


    def drop_electrodes(x):
        B, C, _ = x.shape
        k = torch.randint(0, MAX_DROP_ELECTRODES + 1, (B, 1), device=x.device)
        keep = torch.argsort(torch.rand(B, C, device=x.device), dim=1) >= k      # k random rows dropped
        return x * keep[:, :, None]


    def views(x):
        """The window as it is; a time-shifted noisy copy; a copy with the electrode rows shuffled and a few
        electrodes zeroed (the time axis inside a row is untouched). Originals only when AUGMENT is False."""
        out = [x]
        if AUGMENT:
            t = time_shift(x)
            out.append(t + NOISE_FRAC * t.std(-1, keepdim=True) * torch.randn_like(t))
            order = torch.argsort(torch.rand(x.shape[0], x.shape[1], device=x.device), dim=1)
            out.append(drop_electrodes(x.gather(1, order[:, :, None].expand_as(x))))
        return out


    @torch.no_grad()
    def infer(X, bs=256):
        model.eval()
        out = []
        for i in range(0, len(X), bs):
            with torch.autocast('cuda', dtype=torch.float16, enabled=AMP):
                out.append(model(torch.from_numpy(X[i:i + bs]).to(DEVICE)).float())
        return torch.cat(out)


    fit_idx, val_idx = np.where(FIT)[0], np.where(VAL)[0]
    if MAX_FIT_WINDOWS and len(fit_idx) > MAX_FIT_WINDOWS:
        fit_idx = np.sort(rng.choice(fit_idx, MAX_FIT_WINDOWS, replace=False))
        print(f'fitting on a random {MAX_FIT_WINDOWS:,} of the training windows (MAX_FIT_WINDOWS)')
    Xv, yv = Wtr[0][val_idx], Wtr[1][val_idx]     # validation is read into RAM once
    yv_t = torch.tensor(yv.astype('int64'), device=DEVICE)
    groups = [{'params': head_params, 'lr': LR_HEAD}] + (
        [{'params': adapters, 'lr': LR_ADAPTER}] if adapters else [])
    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    # what gets saved and restored: the adapters, the head and BatchNorm (parameters and running
    # statistics); for the small CNN, everything. The frozen ImageNet convolutions stay in place.
    train_ids = {id(p) for p in adapters + head_params}
    trainable = ({n for n, p in model.named_parameters() if id(p) in train_ids}
                 | {n for n, _ in model.named_buffers()})
    print(f'chance loss {np.log(BATCH):.4f}')

    best, best_state, hist = -1, None, []
    bad, loss_up, vl_best, stopped_at = 0, 0, 1e9, None
    for ep in range(1, EPOCHS + 1):
        on = ep > HEAD_ONLY_EPOCHS
        for p in adapters:
            p.requires_grad_(on)
        model.train()
        tot, t0 = 0.0, time.time()
        perm = rng.permutation(len(fit_idx))
        batches = [fit_idx[perm[i:i + BATCH]] for i in range(0, len(perm) - 7, BATCH)]
        for bi, rows in enumerate(batches, 1):
            if bi in (20, 100) or bi % 500 == 0:
                print(f'    batch {bi}/{len(batches)}  {time.time() - t0:.0f}s  projected epoch '
                      f'{(time.time() - t0) / bi * len(batches) / 60:.1f} min; projected {EPOCHS} epochs '
                      f'{(time.time() - t0) / bi * len(batches) * EPOCHS / 60:.0f} min')
            b = np.sort(rows)                      # sorted rows read faster from disk
            xb = torch.from_numpy(Wtr[0][b]).to(DEVICE).float()
            yb = torch.from_numpy(Wtr[1][b].astype('int64')).to(DEVICE)
            loss = 0.0
            for v in views(xb):
                with torch.autocast('cuda', dtype=torch.float16, enabled=AMP):
                    pred = model(v)
                loss = loss + clip_loss(pred, yb)
            loss = loss / (3 if AUGMENT else 1)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += float(loss) * len(b)
        pv = infer(Xv)
        vl = float(np.mean([float(clip_loss(pv[i:i + BATCH], yv_t[i:i + BATCH]))
                            for i in range(0, len(pv) - 7, BATCH)]))
        r = rank_pct(pv.cpu().numpy(), yv, emb)
        hist.append((ep, tot / len(fit_idx), vl, r))
        last_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()
                      if k in trainable}
        improved = r > best + MIN_DELTA
        if r > best:
            best = r
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()
                          if k in trainable}
            torch.save(best_state, WEIGHTS / f'{RUN_TAG}_best_so_far.pt')   # survives a Colab disconnect
        if ep == 2:
            print('  learning check: ' + ('training loss is below chance, it is learning'
                  if tot / len(fit_idx) < np.log(BATCH) - 0.02 else
                  f'WARNING training loss is still at chance ({np.log(BATCH):.4f}): the model is not learning; stop and loosen'))
        if ep == 1:
            print(f'  epoch 1 took {time.time() - t0:.0f}s; projected total for {EPOCHS} epochs '
                  f'about {(time.time() - t0) * EPOCHS / 60:.0f} min')
        print(f'  epoch {ep:2d}{"" if on else " (head only)"}  train loss {tot / len(fit_idx):.4f}  '
              f'val loss {vl:.4f}  gap {vl - tot / len(fit_idx):+.3f}  val rank {r:.4f}{"  *" if r == best else ""}  '
              f'{time.time() - t0:.0f}s')
        vl_best = min(vl_best, vl)
        bad = 0 if improved else bad + 1
        loss_up = loss_up + 1 if vl > vl_best + DIVERGE_LOSS else 0
        if on and (bad >= PATIENCE or loss_up >= 2):
            stopped_at = ep
            why = (f'no validation-rank gain for {bad} epochs' if bad >= PATIENCE else
                   f'val loss {vl:.4f} is above its minimum {vl_best:.4f} by more than {DIVERGE_LOSS}')
            print(f'  early stop at epoch {ep}: {why}; best was epoch {max(hist, key=lambda h: h[3])[0]}')
            break
    # Only the trained tensors are restored; the frozen convolutions are already in place.
    _ = model.load_state_dict(best_state if KEEP == 'best' else last_state, strict=False)


## Step 4 — score it on held-out people

In [ ]:
if RUN_TRAINING:
    best_ep = max(hist, key=lambda h: h[3])
    vl_min = min(hist, key=lambda h: h[2])
    print(f'best val rank {best_ep[3]:.4f} at epoch {best_ep[0]} (KEEP = {KEEP!r}); '
          f'lowest val loss {vl_min[2]:.4f} at epoch {vl_min[0]}')

    pred = infer(Wte[0]).cpu().numpy()
    r_te = rank_pct(pred, Wte[1], emb)
    a_te, ch = top5(pred, Wte[1], emb)
    agg = {}
    for p_, y_, s_ in zip(pred, Wte[1], Wte[2]):
        agg.setdefault((s_, y_), []).append(p_)
    ks = sorted(agg)
    a_agg, _ = top5(np.stack([np.mean(agg[k_], 0) for k_ in ks]), np.array([k_[1] for k_ in ks]), emb)
    e = None
    print(f'\n  {"seen people, new images":<24} rank {r_te:.4f}   top-5 {a_te:5.2f}% (per-window)'
          f'   {a_agg:5.2f}% (subject-agg)   chance {ch:.2f}%')
    if HOLDOUT_PEOPLE:
        hm = np.isin(Wte[2], [str(p_) for p_ in HOLDOUT_PEOPLE])
        r_h = rank_pct(pred[hm], Wte[1][hm], emb)
        a_h, ch_h = top5(pred[hm], Wte[1][hm], emb)
        agg_h = {}
        for p_, y_, s_ in zip(pred[hm], Wte[1][hm], Wte[2][hm]):
            agg_h.setdefault((s_, y_), []).append(p_)
        ks_h = sorted(agg_h)
        a_hagg, _ = top5(np.stack([np.mean(agg_h[k_], 0) for k_ in ks_h]),
                         np.array([k_[1] for k_ in ks_h]), emb)
        print(f'\n  HELD-OUT {str(HOLDOUT_PEOPLE):<9} rank {r_h:.4f}   '
              f'top-5 {a_h:5.2f}%   {a_hagg:5.2f}% (subject-agg)   chance {ch_h:.2f}%   '
              f'{hm.sum():,} windows, {len(ks_h)} images - never fitted or validated')

    nullv = rank_null(infer(Xv).cpu().numpy(), yv)
    thr = float(np.percentile(nullv, 95))
    climb = best_ep[3] - hist[0][3]
    TRENDING = bool(best_ep[3] > thr and best_ep[0] > 1 and climb > 2 * nullv.std())
    print(f'\nval rank under permuted labels: mean {nullv.mean():.4f}  sd {nullv.std():.4f}  '
          f'95th {thr:.4f}; best {best_ep[3]:.4f} '
          f'({(best_ep[3] - nullv.mean()) / max(nullv.std(), 1e-9):.1f} sd above)')
    if TRENDING and e:
        verdict = 'beats' if a_te > e['top5'] else 'does not beat'
        print(f'Trending, and REVE {verdict} EEGNet on the same data '
              f'({a_te:.2f}% vs {e["top5"]:.2f}%).')
    elif TRENDING:
        print('Trending: it learns something that generalises to unseen images.')
    else:
        print('Not trending: validation rank never clears its own null.')

    torch.save(best_state if KEEP == 'best' else last_state, WEIGHTS / f'{RUN_TAG}_trainable.pt')
    print(f'saved -> {WEIGHTS / f"{RUN_TAG}_trainable.pt"} (adapters and head only)')


## Step 5 — package

Merges the LoRA updates into plain convolution weights, checks the merged network gives the same output, writes the weights, config and submission file, reloads them the way the harness does, and zips only if the checks pass.

In [ ]:
def package(model, tag, test_top5):
    """Write merged weights, config and solver; reload them the way the harness does; zip only if
    the reloaded submission reproduces the model it came from."""
    import importlib.util
    out = BASE / 'submissions' / f'track1_{tag}'
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    plain = to_plain(model).eval()
    torch.save({k: v.cpu() for k, v in plain.state_dict().items()}, out / 'weights.pt')
    (out / 'cnn_config.json').write_text(json.dumps(dict(
        channels=CH, n_outputs=DIM, arch=ARCH, shrink=SHRINK, dropout=DROPOUT,
        trained_on=f'Alljoined-1.6M ({tag})', test_top5=test_top5), indent=2))
    (out / 'submission.py').write_text(SUBMISSION_PY)

    spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    def run(names, X):
        m = mod.Solver().load_model({'ch_names': list(names), 'chs_info': None,
                                     'sfreq': IN_SFREQ, 'submission_dir': out,
                                     'device': DEVICE, 'n_outputs': DIM})
        return m.predict(torch.from_numpy(X.astype('float32'))).cpu().numpy()

    Xs = Wte[0][:8].astype('float32')
    base = run(CH, Xs)
    with torch.no_grad():
        model.eval()
        ref = model(torch.from_numpy(Xs).to(DEVICE)).float().cpu().numpy()
    perm = rng.permutation(len(CH))
    shuffled = run([CH[i] for i in perm], Xs[:, perm])
    wide = run(list(CH) + [f'X{i}' for i in range(31)],
               np.concatenate([Xs, rng.standard_normal((8, 31, Xs.shape[-1])).astype('float32')], 1))
    drop = list(CH[2:])
    part = run(drop, Xs[:, [CH.index(c) for c in drop]])
    checks = [
        ('merged plain network = the trained model', np.allclose(base, ref, atol=1e-3),
         f'max |diff| {np.abs(base - ref).max():.1e}'),
        ('order-independent', np.allclose(base, shuffled, atol=1e-4),
         f'max |diff| {np.abs(base - shuffled).max():.1e}'),
        ('extra electrodes', np.allclose(base, wide, atol=1e-4), 'ignored, output identical'),
        ('missing electrodes', part.shape == base.shape and np.isfinite(part).all(),
         'zero-filled, still runs'),
    ]
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    if not all(c[1] for c in checks):
        print('\nNot zipping: a check failed.')
        return None
    zpath = shutil.make_archive(str(out), 'zip', out)
    print(f'\n{zpath}  ({Path(zpath).stat().st_size / 1e6:.0f} MB)')
    return zpath


if RUN_TRAINING and MAKE_ZIP:
    if TRENDING or globals().get('FORCE_ZIP'):
        package(model, RUN_TAG, a_te)
    else:
        print('Not packaging: the run did not clear its null. Set FORCE_ZIP = True to override.')
